# <font color="#418FDE" size="6.5" uppercase>**Cleaning and Visualization**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Clean missing, duplicated, and inconsistent values in small civil engineering datasets. 
- Create useful features and scaled variables for later machine learning models. 
- Visualize distributions, relationships, time trends, and grid patterns with matplotlib. 


## **1. Data Cleaning Basics**

### **1.1. Missing Values**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_01_01.jpg?v=1783736402" width="250">



>* Missing values can affect engineering conclusions
>* Check their location, frequency, and patterns

>* Interpret what each missing value means
>* Avoid blanket fixes that distort engineering data

>* Choose treatment based on analysis needs
>* Document decisions for transparency and trust



In [ ]:
#@title Python Code - Missing Values

# Missing values need careful engineering interpretation.
# Small datasets can lose important records.
# Flags preserve transparency during cleaning decisions.

# Import pandas for small tabular engineering data.
import pandas as pd

# Create a tiny pavement inspection dataset.
data = {
    "segment_id": ["A1", "A2", "A3", "A4", "A5", "A6"],
    "rut_depth_in": [0.18, None, 0.31, -1.0, 0.22, None],

    "cracking_pct": [8, 12, None, 25, 10, 14],
    "traffic_vpd": [1200, 1350, 1280, None, 1180, 1400],
}

# Convert placeholder values into true missing values.
df = pd.DataFrame(data)
df["rut_depth_in"] = df["rut_depth_in"].replace(-1.0, pd.NA)

# Count missing values before any filling.
missing_counts = df.isna().sum()
print("Missing values by column:")
print(missing_counts.to_string())

# Add flags before estimating missing measurements.
df["rut_depth_was_missing"] = df["rut_depth_in"].isna()
df["cracking_was_missing"] = df["cracking_pct"].isna()

# Fill selected numeric gaps using column medians.
rut_median = df["rut_depth_in"].median()
crack_median = df["cracking_pct"].median()

# Apply simple estimates for classroom demonstration.
df["rut_depth_clean_in"] = df["rut_depth_in"].fillna(rut_median)
df["cracking_clean_pct"] = df["cracking_pct"].fillna(crack_median)

# Summarize the cleaning decisions compactly.
print("\nCleaning summary:")
print(f"Rut depth median used: {rut_median:.2f} inches")
print(f"Cracking median used: {crack_median:.1f} percent")
print(f"Rows kept after cleaning: {len(df)}")

# Show only key columns to avoid clutter.
preview_cols = ["segment_id", "rut_depth_clean_in", "rut_depth_was_missing"]
print("\nCleaned preview:")
print(df[preview_cols].to_string(index=False))



### **1.2. Duplicate Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_01_02.jpg?v=1783736406" width="250">



>* Repeated records can arise from data collection issues
>* Duplicates can bias engineering decisions

>* Check exact, partial, and approximate duplicates
>* Use context to avoid removing valid repeats

>* Keep the most reliable duplicate record
>* Do not remove meaningful repeated patterns



In [ ]:
#@title Python Code - Duplicate Data

# Duplicate rows can bias engineering summaries.
# We create a tiny inspection dataset.
# Then we remove exact and key duplicates.

import pandas as pd
import matplotlib.pyplot as plt

# Build records with exact and conflicting duplicates.
records = [
    {"bridge_id": "B-101", "date": "2026-01-05", "span_ft": 120, "rating": 7, "source": "field"},
    {"bridge_id": "B-101", "date": "2026-01-05", "span_ft": 120, "rating": 7, "source": "field"},

    {"bridge_id": "B-102", "date": "2026-01-06", "span_ft": 95, "rating": 5, "source": "field"},
    {"bridge_id": "B-102", "date": "2026-01-06", "span_ft": 95, "rating": 6, "source": "verified"},
    {"bridge_id": "B-103", "date": "2026-01-07", "span_ft": 140, "rating": 8, "source": "field"},

    {"bridge_id": "B-104", "date": "2026-01-08", "span_ft": 80, "rating": 4, "source": "verified"},
]

# Convert records into a small dataframe.
df = pd.DataFrame(records)

# Count exact duplicate rows first.
exact_count = int(df.duplicated().sum())

# Remove exact copies before key checking.
no_exact = df.drop_duplicates().copy()

# Define fields that identify one inspection.
key_cols = ["bridge_id", "date"]

# Prefer verified records when key duplicates conflict.
source_rank = {"field": 0, "verified": 1}
no_exact["source_rank"] = no_exact["source"].map(source_rank)

# Sort so best records appear last.
sorted_df = no_exact.sort_values(key_cols + ["source_rank"])

# Keep the best record for each inspection key.
clean = sorted_df.drop_duplicates(key_cols, keep="last")

# Remove helper column after cleaning.
clean = clean.drop(columns=["source_rank"])

# Summarize the cleaning effect briefly.
print("Original rows:", len(df))
print("Exact duplicate rows:", exact_count)
print("Rows after cleaning:", len(clean))
print("Mean rating before:", round(df["rating"].mean(), 2))
print("Mean rating after:", round(clean["rating"].mean(), 2))

# Plot ratings before and after duplicate cleaning.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Before", "After"], [df["rating"].mean(), clean["rating"].mean()])
ax.set_ylabel("Mean bridge condition rating")
ax.set_title("Duplicate cleaning changes engineering summaries")
plt.show()



### **1.3. Inconsistent Units**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_01_03.jpg?v=1783736404" width="250">



>* Mixed units can make values misleading
>* Always link measurements to consistent units

>* Check documentation, ranges, and engineering context
>* Investigate outliers for hidden unit differences

>* Convert data to one clear unit system
>* Flag uncertain units and document changes



In [ ]:
#@title Python Code - Inconsistent Units

# Mixed units can hide engineering data problems.
# This example standardizes pipe diameters safely.
# Small tables make each cleaning step visible.

import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny culvert inspection dataset.
records = [
    {"culvert_id": "C1", "diameter": 900, "unit": "mm"},
    {"culvert_id": "C2", "diameter": 36, "unit": "in"},

    {"culvert_id": "C3", "diameter": 1.2, "unit": "m"},
    {"culvert_id": "C4", "diameter": 48, "unit": "in"},
    {"culvert_id": "C5", "diameter": 750, "unit": "mm"},
]

# Load records into a small dataframe.
df = pd.DataFrame(records)

# Define conversion factors to millimeters.
unit_to_mm = {"mm": 1.0, "in": 25.4, "m": 1000.0}

# Convert every diameter into one unit.
df["diameter_mm"] = df["diameter"] * df["unit"].map(unit_to_mm)

# Flag units that cannot be converted.
df["unit_known"] = df["unit"].isin(unit_to_mm)

# Print only compact teaching summaries.
print("Original mixed-unit values:")
print(df[["culvert_id", "diameter", "unit"]].to_string(index=False))

# Show the cleaned standardized values.
print("\nCleaned values in millimeters:")
print(df[["culvert_id", "diameter_mm", "unit_known"]].to_string(index=False))

# Compare misleading raw values with cleaned values.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))

# Plot raw mixed-unit diameters.
axes[0].bar(df["culvert_id"], df["diameter"], color="tomato")
axes[0].set_title("Raw mixed units")
axes[0].set_ylabel("Recorded number")

# Plot standardized millimeter diameters.
axes[1].bar(df["culvert_id"], df["diameter_mm"], color="steelblue")
axes[1].set_title("Standardized units")
axes[1].set_ylabel("Diameter in millimeters")

# Improve spacing before display.
plt.tight_layout()
plt.show()



## **2. Feature Preparation**

### **2.1. Feature Engineering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_02_01.jpg?v=1783736412" width="250">



>* Turn raw data into meaningful model inputs
>* Create features reflecting real engineering conditions

>* Use relevant features available before prediction
>* Derive interpretable variables that reveal patterns

>* Convert categories, dates, and locations into features
>* Check features for meaning, leakage, and clarity



In [ ]:
#@title Python Code - Feature Engineering

# Feature engineering turns raw measurements into useful model inputs.
# This example uses a tiny pavement inspection dataset.
# New columns should reflect engineering meaning clearly.

import pandas as pd
import matplotlib.pyplot as plt

# Create a small civil engineering dataset.
data = {
    "segment": ["A", "B", "C", "D", "E", "F"],
    "year_built": [1998, 2005, 2012, 1990, 2018, 2001],

    "inspection_year": [2024, 2024, 2024, 2024, 2024, 2024],
    "daily_trucks": [420, 310, 180, 560, 140, 390],
    "lanes": [2, 2, 4, 2, 4, 3],

    "pavement_thickness_in": [8.0, 7.5, 10.0, 6.5, 9.0, 7.0],
    "rainfall_mm": [42, 35, 18, 55, 22, 48],
    "condition_score": [68, 74, 86, 52, 90, 63],
}

# Convert the dictionary into a table.
df = pd.DataFrame(data)

# Validate the dataset before creating features.
required = {"year_built", "inspection_year", "daily_trucks", "lanes"}
assert required.issubset(df.columns), "Required columns are missing."

# Create age at inspection time.
df["asset_age_years"] = df["inspection_year"] - df["year_built"]

# Create traffic intensity per lane.
df["trucks_per_lane"] = df["daily_trucks"] / df["lanes"]

# Convert pavement thickness from inches to centimeters.
df["thickness_cm"] = df["pavement_thickness_in"] * 2.54

# Scale selected numeric features manually.
scale_cols = ["asset_age_years", "trucks_per_lane", "rainfall_mm", "thickness_cm"]

# Use min-max scaling for beginner clarity.
for col in scale_cols:
    spread = df[col].max() - df[col].min()
    df[col + "_scaled"] = (df[col] - df[col].min()) / spread

# Print only selected engineered features.
print("Engineered pavement features:")
print(df[["segment", "asset_age_years", "trucks_per_lane", "thickness_cm"]])

# Plot one engineered feature against condition.
plt.figure(figsize=(7, 4))
plt.scatter(df["trucks_per_lane"], df["condition_score"], s=80)

# Label each point by segment.
for _, row in df.iterrows():
    plt.text(row["trucks_per_lane"] + 2, row["condition_score"], row["segment"])

# Add clear engineering labels.
plt.xlabel("Daily trucks per lane")
plt.ylabel("Pavement condition score")
plt.title("Engineered traffic intensity versus pavement condition")

# Improve readability and show the plot.
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **2.2. Scaling Variables**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_02_02.jpg?v=1783736408" width="250">



>* Put different measurements on comparable scales
>* Prevent large numbers from dominating models

>* Transform features for fair model comparison
>* Fit scaling on training data only

>* Scale based on model needs
>* Check outliers, units, and engineering meaning



In [ ]:
#@title Python Code - Scaling Variables

# Scaling prepares engineering variables for modeling.
# Small datasets make transformations easy to inspect.
# Plots reveal comparable ranges after scaling.

# Required libraries are already available in Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny pavement dataset.
data = {
    "segment": ["A", "B", "C", "D", "E", "F"],
    "age_years": [2, 5, 9, 12, 16, 20],
    "traffic_vpd": [3200, 8500, 12000, 18000, 26000, 41000],
    "rut_depth_mm": [3, 6, 9, 13, 18, 25],
}

# Store the records in a dataframe.
df = pd.DataFrame(data)
features = ["age_years", "traffic_vpd", "rut_depth_mm"]

# Validate the selected numeric columns.
assert len(df) > 1
assert df[features].notna().all().all()

# Compute training-style scaling statistics.
means = df[features].mean()
stds = df[features].std(ddof=0)

# Protect against constant columns.
safe_stds = stds.replace(0, 1)
scaled = (df[features] - means) / safe_stds

# Add scaled variables for modeling.
scaled_columns = [name + "_scaled" for name in features]
df[scaled_columns] = scaled.round(2)

# Print a compact teaching summary.
print("Original ranges versus scaled ranges:")
for name in features:
    original_range = df[name].max() - df[name].min()
    scaled_range = df[name + "_scaled"].max() - df[name + "_scaled"].min()
    print(f"{name}: {original_range:.1f} becomes {scaled_range:.2f}")

# Compare original and scaled feature ranges.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Plot original engineering units.
df[features].plot(kind="bar", ax=axes[0])
axes[0].set_title("Original variables")
axes[0].set_xlabel("Road segment index")
axes[0].set_ylabel("Original units")

# Plot standardized variables together.
df[scaled_columns].plot(kind="bar", ax=axes[1])
axes[1].set_title("Scaled variables")
axes[1].set_xlabel("Road segment index")
axes[1].set_ylabel("Standard deviations from mean")

# Improve spacing for the single figure.
plt.tight_layout()
plt.show()



### **2.3. Scaled Variables**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_02_03.jpg?v=1783736410" width="250">



>* Put different measurements on comparable scales
>* Prevent large units from dominating models

>* Preserve patterns while adjusting variable ranges
>* Clean data and handle outliers first

>* Keep scaled data linked to original meaning
>* Document scaling and reuse it consistently



In [ ]:
#@title Python Code - Scaled Variables

# Scaled variables support fair engineering comparisons.
# This example uses small bridge inspection data.
# Original units remain important for interpretation.

# Import common data and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny civil engineering dataset.
data = {
    "bridge_id": ["B1", "B2", "B3", "B4", "B5", "B6"],
    "span_m": [18, 42, 35, 60, 25, 50],

    "traffic_vpd": [4200, 18500, 9600, 24000, 7100, 15500],
    "crack_mm": [0.4, 1.8, 1.1, 2.6, 0.7, 1.5],
    "age_years": [8, 45, 28, 62, 15, 39],
}

# Convert the dictionary into a dataframe.
df = pd.DataFrame(data)

# Select numeric columns for scaling.
features = ["span_m", "traffic_vpd", "crack_mm", "age_years"]
values = df[features].astype(float)

# Check that scaling inputs are usable.
assert values.shape[0] > 1 and values.shape[1] == len(features)

# Compute means and standard deviations.
means = values.mean()
stds = values.std(ddof=0).replace(0, 1)

# Create centered and spread-scaled variables.
z_scaled = (values - means) / stds
z_scaled = z_scaled.add_suffix("_z")

# Compute minimum and maximum values.
mins = values.min()
ranges = (values.max() - mins).replace(0, 1)

# Create zero-to-one scaled variables.
minmax_scaled = (values - mins) / ranges
minmax_scaled = minmax_scaled.add_suffix("_minmax")

# Combine original and scaled variables.
prepared = pd.concat([df, z_scaled, minmax_scaled], axis=1)

# Print a compact teaching summary.
print("Original traffic range:", int(values["traffic_vpd"].min()), "to", int(values["traffic_vpd"].max()))
print("Scaled traffic range:", round(minmax_scaled["traffic_vpd_minmax"].min(), 2), "to", round(minmax_scaled["traffic_vpd_minmax"].max(), 2))
print("Original crack range:", values["crack_mm"].min(), "to", values["crack_mm"].max())
print("Scaled crack range:", round(minmax_scaled["crack_mm_minmax"].min(), 2), "to", round(minmax_scaled["crack_mm_minmax"].max(), 2))

# Show selected columns without printing everything.
print(prepared[["bridge_id", "traffic_vpd_minmax", "crack_mm_minmax"]].round(2).to_string(index=False))

# Plot original and scaled traffic values.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(df["bridge_id"], df["traffic_vpd"], color="steelblue")
axes[0].set_title("Original traffic")

axes[0].set_ylabel("Vehicles per day")
axes[1].bar(df["bridge_id"], minmax_scaled["traffic_vpd_minmax"], color="darkorange")
axes[1].set_title("Scaled traffic")
axes[1].set_ylabel("Zero-to-one scale")

# Improve spacing before displaying the figure.
plt.tight_layout()
plt.show()



## **3. matplotlib Plots**

### **3.1. Distribution Plots**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_03_01.jpg?v=1783736414" width="250">



>* Show how engineering measurements are spread
>* Reveal patterns for cleaning and decisions

>* Choose histogram intervals carefully
>* Label units, titles, and thresholds clearly

>* Compare group spreads, medians, and outliers
>* Connect distribution patterns to engineering decisions



In [ ]:
#@title Python Code - Distribution Plots

# Distribution plots reveal engineering measurement spread.
# Histograms help identify skew and outliers.
# Reference lines connect plots to specifications.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Make random values reproducible.
rng = np.random.default_rng(42)

# Simulate concrete strengths in megapascals.
strength_mpa = rng.normal(34, 4, 45)

# Add a few low strength observations.
low_strengths = np.array([22, 24, 25])

# Combine normal and low observations.
strength_mpa = np.concatenate([strength_mpa, low_strengths])

# Validate the dataset before plotting.
assert strength_mpa.size > 10

# Define the design strength threshold.
design_mpa = 28

# Count samples below the threshold.
below_count = np.sum(strength_mpa < design_mpa)

# Print a compact engineering summary.
print(f"Samples analyzed: {strength_mpa.size}")
print(f"Mean strength: {strength_mpa.mean():.1f} MPa")
print(f"Below design strength: {below_count} samples")

# Create one histogram figure.
fig, ax = plt.subplots(figsize=(7, 4))

# Plot the strength distribution.
ax.hist(strength_mpa, bins=8, color="steelblue", edgecolor="black")

# Add the design requirement line.
ax.axvline(design_mpa, color="crimson", linewidth=2, label="Design limit")

# Label the engineering plot clearly.
ax.set_title("Concrete Compressive Strength Distribution")
ax.set_xlabel("Compressive strength (MPa)")
ax.set_ylabel("Number of test cylinders")

# Add grid and legend for readability.
ax.grid(True, alpha=0.3)
ax.legend()

# Display the completed distribution plot.
plt.show()



### **3.2. Time Trend Plots**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_03_02.jpg?v=1783736416" width="250">



>* Show measurements changing over time
>* Reveal trends, cycles, spikes, and behavior

>* Choose readable axes and appropriate scales
>* Use lines, markers, and legends carefully

>* Spot data errors through time patterns
>* Use trends to guide reliable modeling



In [ ]:
#@title Python Code - Time Trend Plots

# Time trend plots reveal engineering changes.
# Small datasets make cleaning issues visible.
# Matplotlib connects observations across ordered time.

# Import common data and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small daily bridge dataset.
dates = pd.date_range("2026-07-01", periods=14, freq="D")
strain = [102, 104, 103, np.nan, 108, 109, 109]
strain = strain + [112, 250, 115, 116, 116, 118, 119]

# Store readings in a simple dataframe.
data = pd.DataFrame({"date": dates, "strain_microstrain": strain})
assert len(data) == 14 and data.shape[1] == 2

# Add a duplicate row to mimic logging errors.
duplicate_row = data.iloc[[10]].copy()
raw_data = pd.concat([data, duplicate_row], ignore_index=True)

# Clean duplicated dates before plotting trends.
clean_data = raw_data.drop_duplicates(subset="date", keep="first")
clean_data = clean_data.sort_values("date").reset_index(drop=True)

# Fill one missing sensor value by interpolation.
clean_data["strain_clean"] = clean_data["strain_microstrain"].interpolate()
clean_data["rolling_mean"] = clean_data["strain_clean"].rolling(3).mean()

# Print a short cleaning summary.
print("Raw rows:", len(raw_data))
print("Clean rows:", len(clean_data))
print("Missing after interpolation:", int(clean_data["strain_clean"].isna().sum()))

# Build one clear time trend plot.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(clean_data["date"], clean_data["strain_clean"], marker="o")
ax.plot(clean_data["date"], clean_data["rolling_mean"], linestyle="--")

# Highlight a suspicious spike for discussion.
spike = clean_data["strain_clean"].idxmax()
ax.scatter(clean_data.loc[spike, "date"], clean_data.loc[spike, "strain_clean"], s=90)
ax.set_title("Bridge Strain Time Trend After Basic Cleaning")
ax.set_xlabel("Date")

# Label units and improve readability.
ax.set_ylabel("Strain, microstrain")
ax.legend(["Cleaned daily reading", "Three-day rolling mean", "Possible spike"])
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **3.3. Grid Pattern Plots**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_02/Lecture_B/image_03_03.jpg?v=1783736418" width="250">



>* Map engineering measurements across physical locations
>* Spot clustered problem areas needing investigation

>* Check whether data are regular or irregular
>* Use interpolation and color scales carefully

>* Label plots clearly for engineering context
>* Interpret patterns with judgment and caution



In [ ]:
#@title Python Code - Grid Pattern Plots

# Grid plots reveal spatial engineering patterns.
# Small arrays keep examples beginner friendly.
# Color bars connect values with units.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Create repeatable bridge deck coordinates.
x_feet = np.linspace(0, 60, 13)
y_feet = np.linspace(0, 24, 7)

# Build coordinate grids for plotting.
X_grid, Y_grid = np.meshgrid(x_feet, y_feet)

# Simulate chloride concentration across deck.
base_level = 0.18 + 0.004 * X_grid
curb_effect = 0.10 * np.exp(-((Y_grid - 22) ** 2) / 18)
joint_effect = 0.16 * np.exp(-((X_grid - 48) ** 2) / 80)

# Combine effects into one surface.
chloride = base_level + curb_effect + joint_effect
chloride = np.round(chloride, 3)

# Validate grid and value shapes.
assert X_grid.shape == chloride.shape
assert Y_grid.shape == chloride.shape

# Print a short engineering summary.
print("Grid size:", chloride.shape)
print("Minimum chloride:", chloride.min(), "percent")
print("Maximum chloride:", chloride.max(), "percent")

# Create one grid pattern plot.
fig, ax = plt.subplots(figsize=(8, 4.8))

# Draw filled contours for spatial variation.
plot = ax.contourf(X_grid, Y_grid, chloride, levels=12, cmap="YlOrRd")

# Add measured grid points for context.
ax.scatter(X_grid, Y_grid, color="black", s=12, alpha=0.55)

# Label the engineering plot clearly.
ax.set_title("Bridge Deck Chloride Grid Pattern")
ax.set_xlabel("Distance along deck (ft)")
ax.set_ylabel("Distance across deck (ft)")

# Add a color bar with units.
colorbar = fig.colorbar(plot, ax=ax)
colorbar.set_label("Chloride concentration (%)")

# Improve spacing before display.
fig.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Cleaning and Visualization**</font>


In this lecture, you learned to:
- Clean missing, duplicated, and inconsistent values in small civil engineering datasets. 
- Create useful features and scaled variables for later machine learning models. 
- Visualize distributions, relationships, time trends, and grid patterns with matplotlib. 

In the next Module (Module 3), we will go over 'None'